# Structured Streaming (bonus): scoring incoming posts in micro-batches and raising an alert when the share of distress posts spikes

In [ ]:
from datetime import datetime, timezone

import matplotlib.pyplot as plt
from pyspark.sql import functions as F

import GX_utils as U

spark = U.get_spark("GX_Streaming")
U.setup_storage(spark)

model_path = U.work_path("models", "emotion_classifier")
landing_dir = U.work_path("stream", "landing")
checkpoint_dir = U.work_path("stream", "checkpoints")
scored_target = U.table_location("silver", "stream_scored_posts")
dashboard_target = U.table_location("gold", "stream_distress_dashboard")

for path in (landing_dir, checkpoint_dir):
    U.reset_path(path)
U.drop_table(spark, "silver", "stream_scored_posts")
U.drop_table(spark, "gold", "stream_distress_dashboard")

## 1. Simulated feed: 12 hours of unseen test posts as JSON files, plus an injected one-hour distress spike (08:00–09:00) to test the alert

In [ ]:
start = int(datetime(2026, 12, 1, tzinfo=timezone.utc).timestamp())
hours, n_files = 12, 24

test_posts = (U.read_table(spark, "silver", "mental_health")
    .join(U.read_table(spark, "gold", "ml_test_ids"), "ID")
    .select("ID", F.col("Text").alias("text"), "Emotion"))

baseline_feed = test_posts.withColumn(
    "event_time", F.timestamp_seconds(F.lit(start) + (F.rand(U.SEED) * hours * 3600).cast("long")))
spike_feed = (test_posts.filter(F.col("Emotion").isin(U.DISTRESS_EMOTIONS))
    .sample(fraction=0.3, seed=U.SEED)
    .withColumn("ID", F.col("ID") + 10_000_000)
    .withColumn("event_time", F.timestamp_seconds(F.lit(start + 8 * 3600) + (F.rand(U.SEED + 1) * 3600).cast("long"))))

feed = baseline_feed.unionByName(spike_feed)
(feed.select("ID", "text", "event_time")
    .repartitionByRange(n_files, "event_time")
    .write.mode("overwrite").json(landing_dir))

print(f"posts in feed: {feed.count():,} (spike: {spike_feed.count():,}) written as ~{n_files} files to {landing_dir}")

## 2. Stream 1 (Bronze → Silver): read one file per micro-batch, clean the text and score it with the saved MLlib pipeline

In [ ]:
feed_schema = "ID long, text string, event_time timestamp"
incoming = (spark.readStream.schema(feed_schema)
    .option("maxFilesPerTrigger", 1)
    .json(landing_dir)
    .withColumn("cleaned_text", U.clean_text(F.col("text"))))

distress_idx = [U.EMOTION_LABELS.index(e) for e in U.DISTRESS_EMOTIONS]
is_databricks = U.IS_DATABRICKS


# Self-contained so it can be serialised to serverless workers.
def score_batch(batch_df, batch_id):
    from pyspark.ml import PipelineModel
    from pyspark.ml.functions import vector_to_array
    from pyspark.sql import functions as F

    probabilities = vector_to_array(F.col("probability"))
    scored = (PipelineModel.load(model_path).transform(batch_df)
        .withColumn("distress_score", sum(probabilities[i] for i in distress_idx))
        .select("ID", "event_time", "text", "predicted_emotion", "distress_score",
                F.lit(batch_id).alias("micro_batch")))
    if is_databricks:
        scored.write.mode("append").saveAsTable(scored_target)
    else:
        scored.write.mode("append").parquet(scored_target)


scoring_query = (incoming.writeStream
    .foreachBatch(score_batch)
    .option("checkpointLocation", f"{checkpoint_dir}/scoring")
    .trigger(availableNow=True)
    .start())
scoring_query.awaitTermination()

print(f"micro-batches: {len(scoring_query.recentProgress)} | "
      f"rows processed: {sum(p['numInputRows'] for p in scoring_query.recentProgress):,}")

## 3. Stream 2 (Silver → Gold): 30-minute event-time windows with distress share, average distress score and high-risk post count

In [ ]:
scored_schema = ("ID long, event_time timestamp, text string, predicted_emotion string, "
                 "distress_score double, micro_batch long")
scored_stream = (spark.readStream.table(scored_target) if U.IS_DATABRICKS
                 else spark.readStream.schema(scored_schema).parquet(scored_target))

window_stats = (scored_stream
    .groupBy(F.window("event_time", "30 minutes"))
    .agg(F.count("*").alias("posts"),
         F.sum(F.col("predicted_emotion").isin(U.DISTRESS_EMOTIONS).cast("int")).alias("distress_posts"),
         F.avg("distress_score").alias("avg_distress_score"),
         F.sum((F.col("distress_score") > 0.8).cast("int")).alias("high_risk_posts")))


def publish_dashboard(batch_df, batch_id):
    from pyspark.sql import functions as F

    out = batch_df.select(F.col("window.start").alias("window_start"), F.col("window.end").alias("window_end"),
                          "posts", "distress_posts", "avg_distress_score", "high_risk_posts",
                          (F.col("distress_posts") / F.col("posts")).alias("distress_share"))
    if is_databricks:
        out.write.mode("overwrite").saveAsTable(dashboard_target)
    else:
        out.write.mode("overwrite").parquet(dashboard_target)


dashboard_query = (window_stats.writeStream
    .outputMode("complete")
    .foreachBatch(publish_dashboard)
    .option("checkpointLocation", f"{checkpoint_dir}/dashboard")
    .trigger(availableNow=True)
    .start())
dashboard_query.awaitTermination()

## 4. Alerting: flag windows whose distress share exceeds the median by more than 3 robust standard deviations (MAD)

In [ ]:
dashboard = U.read_table(spark, "gold", "stream_distress_dashboard").orderBy("window_start").toPandas()

median_share = dashboard["distress_share"].median()
mad = (dashboard["distress_share"] - median_share).abs().median()
threshold = median_share + 3 * 1.4826 * mad
dashboard["alert"] = dashboard["distress_share"] > threshold

print(f"baseline distress share: {median_share:.1%} | alert threshold: {threshold:.1%}")
dashboard[["window_start", "posts", "distress_posts", "distress_share", "high_risk_posts", "alert"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dashboard["window_start"], dashboard["distress_share"], marker="o", label="distress share")
ax.axhline(threshold, color="crimson", ls="--", label="alert threshold")
alerts = dashboard[dashboard["alert"]]
ax.scatter(alerts["window_start"], alerts["distress_share"], color="crimson", s=80, zorder=3, label="alert")
ax.set_ylabel("Share of posts predicted as distress"); ax.set_title("Live distress monitor (30-minute windows)", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Triage queue: highest-risk posts inside alert windows, for review by a human counsellor (a prioritisation aid, not a diagnosis)

In [ ]:
alert_windows = spark.createDataFrame(alerts[["window_start", "window_end"]]) if len(alerts) else None

if alert_windows is None:
    print("no alert windows")
else:
    (U.read_table(spark, "silver", "stream_scored_posts")
        .join(alert_windows, (F.col("event_time") >= F.col("window_start")) & (F.col("event_time") < F.col("window_end")))
        .orderBy(F.desc("distress_score"))
        .select("event_time", "predicted_emotion", F.round("distress_score", 3).alias("distress_score"),
                F.substring("text", 1, 90).alias("snippet"))
        .show(10, truncate=False))

In [ ]:
print(f"{int(dashboard['alert'].sum())} of {len(dashboard)} windows raised an alert: "
      f"{', '.join(alerts['window_start'].dt.strftime('%H:%M')) or 'none'} (spike injected at 08:00-09:00).")